In [5]:
import os

BASE_PATH = "../HousePrices_TV1/results/member1/processed"

print(os.path.exists(BASE_PATH))
print(os.listdir(BASE_PATH))

True
['test_ids.csv', 'X_test_full.csv', 'X_test_top_20.csv', 'X_test_top_40.csv', 'X_test_top_60.csv', 'X_train_full.csv', 'X_train_top_20.csv', 'X_train_top_40.csv', 'X_train_top_60.csv', 'X_val_full.csv', 'X_val_top_20.csv', 'X_val_top_40.csv', 'X_val_top_60.csv', 'y_train.csv', 'y_val.csv']


In [6]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error

BASE_PATH = "../HousePrices_TV1/results/member1/processed"

In [8]:
datasets = {}

for feature_set in ["full", "top_20", "top_40", "top_60"]:
    datasets[feature_set] = {
        "X_train": pd.read_csv(
            f"{BASE_PATH}/X_train_{feature_set}.csv"
        ),
        "X_val": pd.read_csv(
            f"{BASE_PATH}/X_val_{feature_set}.csv"
        ),
        "X_test": pd.read_csv(
            f"{BASE_PATH}/X_test_{feature_set}.csv"
        )
    }

y_train = pd.read_csv(f"{BASE_PATH}/y_train.csv").squeeze()
y_val = pd.read_csv(f"{BASE_PATH}/y_val.csv").squeeze()
test_ids = pd.read_csv(f"{BASE_PATH}/test_ids.csv")

In [9]:
for name, data in datasets.items():
    print(f"\n===== {name.upper()} =====")
    print("X_train:", data["X_train"].shape)
    print("X_val:  ", data["X_val"].shape)
    print("X_test: ", data["X_test"].shape)

print("\ny_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("test_ids:", test_ids.shape)


===== FULL =====
X_train: (1168, 279)
X_val:   (292, 279)
X_test:  (1459, 279)

===== TOP_20 =====
X_train: (1168, 20)
X_val:   (292, 20)
X_test:  (1459, 20)

===== TOP_40 =====
X_train: (1168, 40)
X_val:   (292, 40)
X_test:  (1459, 40)

===== TOP_60 =====
X_train: (1168, 60)
X_val:   (292, 60)
X_test:  (1459, 60)

y_train: (1168,)
y_val:   (292,)
test_ids: (1459, 1)


In [10]:
from sklearn.linear_model import LinearRegression

In [11]:
def evaluate_model(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)

    predictions = model.predict(X_val)
    predictions = np.maximum(predictions, 0)

    rmse_log = np.sqrt(
        mean_squared_error(
            np.log1p(y_val),
            np.log1p(predictions)
        )
    )

    return rmse_log

In [14]:
#Chạy thử Linear Regression với Full

model = LinearRegression()

rmse = evaluate_model(
    model,
    datasets["full"]["X_train"],
    y_train,
    datasets["full"]["X_val"],
    y_val
)

print("Linear Regression - Full")
print("Log RMSE:", rmse)

Linear Regression - Full
Log RMSE: 0.16090043738538756


In [15]:
#Chạy thử Linear Regression với Top20

model = LinearRegression()

rmse = evaluate_model(
    model,
    datasets["top_20"]["X_train"],
    y_train,
    datasets["top_20"]["X_val"],
    y_val
)

print("Linear Regression - Top20")
print("Log RMSE:", rmse)

Linear Regression - Top20
Log RMSE: 0.17447858948524192


In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [17]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}

In [18]:
results = []

for model_name, model in models.items():
    for feature_set in ["full", "top_20", "top_40", "top_60"]:

        X_train = datasets[feature_set]["X_train"]
        X_val = datasets[feature_set]["X_val"]

        rmse = evaluate_model(
            model,
            X_train,
            y_train,
            X_val,
            y_val
        )

        results.append({
            "Model": model_name,
            "Feature Set": feature_set,
            "Log RMSE": rmse
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Log RMSE",
    ascending=True
).reset_index(drop=True)

results_df

,Model,Feature Set,Log RMSE
0,Random Forest,full,0.151819
1,Random Forest,top_60,0.152101
2,Random Forest,top_40,0.157294
3,Linear Regression,top_60,0.158576
4,Linear Regression,full,0.160900
5,Random Forest,top_20,0.164018
6,Linear Regression,top_20,0.174479
7,Linear Regression,top_40,0.174945
8,Decision Tree,top_60,0.208696
9,Decision Tree,top_40,0.211511


In [22]:
from xgboost import XGBRegressor

xgb_results = []

for feature_set in ["full", "top_20", "top_40", "top_60"]:
    X_train = datasets[feature_set]["X_train"]
    X_val = datasets[feature_set]["X_val"]

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    score = evaluate_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val
    )

    xgb_results.append({
        "Model": "XGBoost",
        "Feature Set": feature_set,
        "Log RMSE": score
    })

xgb_results_df = pd.DataFrame(xgb_results)
xgb_results_df.sort_values("Log RMSE")

,Model,Feature Set,Log RMSE
0,XGBoost,full,0.137608
3,XGBoost,top_60,0.142951
2,XGBoost,top_40,0.150651
1,XGBoost,top_20,0.162683


In [27]:
best_baseline = {
    "Model": "XGBoost",
    "Feature Set": "full",
    "Log RMSE": 0.137608
}

best_baseline

{'Model': 'XGBoost', 'Feature Set': 'full', 'Log RMSE': 0.137608}

In [28]:
from sklearn.model_selection import RandomizedSearchCV

X_train = datasets["full"]["X_train"]
X_val = datasets["full"]["X_val"]

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [300, 500, 700, 1000],
    "learning_rate": [0.02, 0.03, 0.05, 0.08],
    "max_depth": [3, 4, 5, 6],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.05, 0.1]
}

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(
    X_train,
    np.log1p(y_train)
)

print("Best parameters:")
print(random_search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best parameters:
{'subsample': 0.9, 'n_estimators': 700, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.08, 'gamma': 0, 'colsample_bytree': 1.0}


In [29]:
best_xgb = XGBRegressor(
    subsample=0.9,
    n_estimators=700,
    min_child_weight=5,
    max_depth=3,
    learning_rate=0.08,
    gamma=0,
    colsample_bytree=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

best_xgb.fit(X_train, y_train)

val_pred = best_xgb.predict(X_val)
val_pred = np.maximum(val_pred, 0)

tuned_rmse = np.sqrt(
    mean_squared_error(
        np.log1p(y_val),
        np.log1p(val_pred)
    )
)

print("Baseline XGBoost RMSE:", best_baseline["Log RMSE"])
print("Tuned XGBoost RMSE:", tuned_rmse)
print("Improvement:", best_baseline["Log RMSE"] - tuned_rmse)

Baseline XGBoost RMSE: 0.137608
Tuned XGBoost RMSE: 0.13542563194738774
Improvement: 0.0021823680526122657


In [30]:
X_full_train = pd.concat(
    [X_train, X_val],
    axis=0,
    ignore_index=True
)

y_full_train = pd.concat(
    [y_train, y_val],
    axis=0,
    ignore_index=True
)

X_test = datasets["full"]["X_test"]

print("Training shape:", X_full_train.shape)
print("Test shape:", X_test.shape)

Training shape: (1460, 279)
Test shape: (1459, 279)


In [31]:
final_model = XGBRegressor(
    subsample=0.9,
    n_estimators=700,
    min_child_weight=5,
    max_depth=3,
    learning_rate=0.08,
    gamma=0,
    colsample_bytree=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    X_full_train,
    y_full_train
)

test_pred = final_model.predict(X_test)
test_pred = np.maximum(test_pred, 0)

print("Prediction completed.")
print(test_pred[:10])

Prediction completed.
[122677.27  162546.94  185429.33  199122.84  175351.98  169811.52
 182319.38  170278.89  180428.78  125533.766]


In [32]:
submission = pd.DataFrame({
    "Id": test_ids["Id"],
    "SalePrice": test_pred
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("\nShape:", submission.shape)
print("\nFile đã tạo: submission.csv")

     Id      SalePrice
0  1461  122677.273438
1  1462  162546.937500
2  1463  185429.328125
3  1464  199122.843750
4  1465  175351.984375

Shape: (1459, 2)

File đã tạo: submission.csv


In [33]:
print(submission.isnull().sum())
print(submission["SalePrice"].min())
print(submission["SalePrice"].max())

Id           0
SalePrice    0
dtype: int64
29627.977
554676.2


In [34]:
{
    "n_estimators": 700,
    "learning_rate": 0.08,
    "max_depth": 3,
    "min_child_weight": 5,
    "subsample": 0.9,
    "colsample_bytree": 1.0,
    "gamma": 0
}

{'n_estimators': 700,
 'learning_rate': 0.08,
 'max_depth': 3,
 'min_child_weight': 5,
 'subsample': 0.9,
 'colsample_bytree': 1.0,
 'gamma': 0}

In [35]:
final_result = {
    "Best Model": "XGBoost Tuned",
    "Feature Set": "Full",
    "Validation Log RMSE": 0.135426,
    "Kaggle Score": 0.13357
}

final_result

{'Best Model': 'XGBoost Tuned',
 'Feature Set': 'Full',
 'Validation Log RMSE': 0.135426,
 'Kaggle Score': 0.13357}